In [ ]:
import random
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

In [ ]:
# Re-implement the "meeting" topology algorithm as a reference model for testing
# (steps 1-3 exactly as in the description)

from typing import Dict, List, Optional, Callable, Set

class MeetingTopologyReference:
    def __init__(
        self,
        size: int,
        z_star: int,
        n_steps: int,
        npr0: int,
        nmr1: int,
        ne_gamma: int,
        gamma: float = 1.0,
        create_object_method: Optional[Callable[[int], object]] = None,
        seed: Optional[int] = 123,
        initial_edges: Optional[List[tuple[int,int]]] = None,
    ):
        self.size = size
        self.z_star = z_star
        self.n_steps = n_steps
        self.npr0 = npr0
        self.nmr1 = nmr1
        self.ne_gamma = ne_gamma
        self.gamma = gamma
        self.create_object_method = create_object_method or (lambda x: x)
        self.seed = seed
        self.initial_edges = initial_edges or []

    def create(self) -> Dict[int, List[object]]:
        rng = random.Random(self.seed)

        adj: List[Set[int]] = [set() for _ in range(self.size)]

        def deg(i: int) -> int:
            return len(adj[i])

        def can_add(i: int) -> bool:
            return deg(i) < self.z_star

        def try_add(u: int, v: int) -> bool:
            if u == v:
                return False
            if v in adj[u]:
                return False
            if not (can_add(u) and can_add(v)):
                return False
            adj[u].add(v); adj[v].add(u)
            return True

        def try_remove(u: int, v: int) -> bool:
            if v not in adj[u]:
                return False
            adj[u].remove(v); adj[v].remove(u)
            return True

        # init edges (optional)
        for u, v in self.initial_edges:
            try_add(u, v)

        # helper: weighted sampling with replacement
        def weighted_choices(weights: List[float], k: int) -> List[int]:
            total = sum(weights)
            if total <= 0 or k <= 0:
                return []
            population = list(range(self.size))
            return rng.choices(population, weights=weights, k=k)

        for _ in range(self.n_steps):

            # (1) npr0 random pairs uniformly
            for _ in range(self.npr0):
                u, v = rng.sample(range(self.size), 2)
                try_add(u, v)

            # (2) nmr1 vertices with prob ∝ z_i(z_i-1)
            w2 = [float(deg(i) * (deg(i) - 1)) for i in range(self.size)]
            chosen = weighted_choices(w2, self.nmr1)

            for i in chosen:
                neigh = list(adj[i])
                if len(neigh) < 2:
                    continue
                a, b = rng.sample(neigh, 2)
                try_add(a, b)

            # (3) ne_gamma vertices with prob ∝ z_i^gamma ; remove edge to random neighbor
            w3 = [float((deg(i) ** self.gamma) if deg(i) > 0 else 0.0) for i in range(self.size)]
            chosen = weighted_choices(w3, self.ne_gamma)

            for i in chosen:
                if not adj[i]:
                    continue
                j = rng.choice(tuple(adj[i]))
                try_remove(i, j)

        return {i: [self.create_object_method(j) for j in sorted(adj[i])] for i in range(self.size)}

In [ ]:
# Import the implementation under test.
# Update the import path to match your project structure.
#
# Example:
# from your_package.MeetingTopology import MeetingTopology

try:
    from MeetingTopology import MeetingTopology  # <-- change this
except Exception as e:
    MeetingTopology = None
    print("Update the import above to your project path. Import error was:", repr(e))

In [ ]:
# Deterministic equality test vs reference reimplementation

params = dict(
    size=200,
    z_star=20,
    n_steps=300,
    npr0=40,
    nmr1=25,
    ne_gamma=15,
    gamma=1.0,
    seed=123,
)

ref = MeetingTopologyReference(
    size=params["size"],
    z_star=params["z_star"],
    n_steps=params["n_steps"],
    npr0=params["npr0"],
    nmr1=params["nmr1"],
    ne_gamma=params["ne_gamma"],
    gamma=params["gamma"],
    seed=params["seed"],
)

ref_graph = ref.create()

if MeetingTopology is None:
    print("Skipping test: MeetingTopology not imported yet.")
else:
    # IMPORTANT: if your MeetingTopology uses Python's global random module,
    # seed it before calling create() so the run is reproducible.
    random.seed(params["seed"])

    impl = MeetingTopology(
        size=params["size"],
        z_star=params["z_star"],
        n_steps=params["n_steps"],
        npr0=params["npr0"],
        nmr1=params["nmr1"],
        ne_gamma=params["ne_gamma"],
        gamma=params["gamma"],
    )

    impl_graph = impl.create()

    # Compare as sets (order-independent)
    ok = True
    for i in range(params["size"]):
        if set(ref_graph[i]) != set(impl_graph[i]):
            ok = False
            print(f"Mismatch at node {i}: ref={sorted(ref_graph[i])[:15]}..., impl={sorted(impl_graph[i])[:15]}...")
            break

    assert ok, "Implementation graph differs from reference reimplementation"
    print("✅ Implementation matches reference for the chosen seed & params.")

In [ ]:
# Build a NetworkX graph from the implementation (or reference if impl not available)
graph = ref_graph if MeetingTopology is None else impl_graph

G = nx.Graph()
for u, neighs in graph.items():
    for v in neighs:
        if u != v:
            G.add_edge(u, v)

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
print("Min degree:", min(dict(G.degree()).values()), "Max degree:", max(dict(G.degree()).values()), "Avg degree:", sum(dict(G.degree()).values())/G.number_of_nodes())

In [ ]:
# Degree distribution plots (like in the attached ScaleFreeTopologyTest)

deg_counts = Counter(dict(G.degree()).values())
degrees = sorted(deg_counts.keys())
freqs = [deg_counts[d] for d in degrees]

plt.figure()
plt.loglog(degrees, freqs, marker='o')
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution (log-log)")
plt.tight_layout()
plt.show()

plt.figure()
plt.plot(degrees, freqs, marker='o')
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.title("Degree Distribution (linear)")
plt.tight_layout()
plt.show()